# Finding tumors across the circulatory system, wherever they're described

*Part 2 of 3. Picks up from [Part 1](#), where I confirmed `sex` is recorded consistently enough across sources to filter on with a wildcard. This part stands on its own if you're starting here -- the only assumption is that you're comfortable with basic `get_subject_data()` filter strings.*

Now for the harder part of this cohort: I want any diagnosis connected to the circulatory system -- not just things literally labeled "circulatory system," but anything that falls under it: veins, blood, platelets, and so on. I don't want to write out every possible sub-term myself and hope I didn't miss one.

In [ ]:
from itables import init_notebook_mode, show
init_notebook_mode(all_interactive=True)
import itables.options as opt
opt.classes="display nowrap compact"
opt.buttons=["copyHtml5", "csvHtml5", "excelHtml5"]
opt.maxBytes=0
opt.columnDefs = [{"className": "dt-left", "targets": "_all"}]
# Force light mode regardless of system/browser dark-mode preference --
# recent DataTables versions auto-detect prefers-color-scheme, which otherwise
# silently flips tables to a dark theme on dark-mode systems.
opt.css = """
.itables, .dataTable { color-scheme: light; }
"""

Before I search, it's worth knowing something about how CDA handles a column like `anatomic_site`. Harmonized columns in CDA are backed by an ontology: every value carries its own synonyms, and its position in a hierarchy of broader and narrower terms.

That means searching for a broad term like `circulatory system` doesn't only match rows tagged exactly `circulatory system`. It also matches:

- any synonym for `circulatory system`
- anything more specific that "rolls up" to it in the ontology -- `vein`, `blood`, `platelet`, and so on

This happens automatically, every time, for any harmonized column. I don't have to turn anything on for it -- this is different from the wildcard trick in Part 1, which I had to choose to use.

In [ ]:
from cdapython import get_subject_data

circulatory_subjects = get_subject_data(
    match_all=['sex = F*', 'anatomic_site = circulatory system'],
    data_source=['GDC', 'PDC', 'IDC', 'GC', 'ICDC', 'CTDC']
)
circulatory_subjects

Scroll through the `anatomic_site` column in these results and something jumps out: not every row actually says "circulatory system." Some say "vein." Some say "blood." Some say "platelet." I searched for one term, and got back everything underneath it in the ontology, because that hierarchy is baked into how this column is harmonized -- not because I asked for anything extra.

That's useful on its own, but if I want to know *why* a particular row matched -- an exact hit, a synonym, or a hierarchy rollup, and from which term -- I can ask CDA to show its work with `add_extras`.

In [ ]:
circulatory_subjects_annotated = get_subject_data(
    match_all=['sex = F*', 'anatomic_site = circulatory system'],
    data_source=['GDC', 'PDC', 'IDC', 'GC', 'ICDC', 'CTDC'],
    add_extras=['all']
)
circulatory_subjects_annotated

Notice this is the exact same set of rows as before -- same count, same subjects. `add_extras` doesn't change *what* matched, only what CDA tells me about *why* it matched. Look at the new columns: for a row where `anatomic_site` reads "vein," there's now a column showing that it matched because "vein" rolls up to "circulatory system" in the ontology. For a row with a synonym like "cardiovascular system," the new column shows it matched as a synonym instead.

This is a bigger deal than it might look like at first glance. Before this release, finding every specimen touching the circulatory system meant already knowing every specific term any data source might use for it -- vein, artery, blood, platelet, cardiovascular system -- and listing every one myself, hoping I hadn't missed a term some source used that I hadn't thought of. Now I search one broad, human term, and the ontology does that enumeration for me, automatically, across every source CDA covers.

I now have my full cohort. In [Part 3](#), I check how much I should trust it -- specifically, whether every source contributed as much data as I'd expect, or whether some are just running on older extractions.

A matching `get_file_data()` call takes the same arguments, if file-level results are what I need instead of subject-level.